In [ ]:
# This would be a Jupyter notebook - here's the content as a Python script

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Load data
df = pd.read_csv('../data/dataset_phishing.csv')
print("Dataset Info:")
print(df.info())
print(f"\nDataset Shape: {df.shape}")
print(f"\nTarget Distribution:\n{df['status'].value_counts()}")

# Basic statistics
print("\nBasic Statistics:")
print(df.describe())

# Check for missing values
print("\nMissing Values:")
print(df.isnull().sum().sum())

# Visualize target distribution
plt.figure(figsize=(10, 6))
df['status'].value_counts().plot(kind='bar')
plt.title('Phishing vs Legitimate URLs')
plt.xlabel('Status')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

# Correlation analysis (for numerical features)
numeric_cols = df.select_dtypes(include=[np.number]).columns
correlation_matrix = df[numeric_cols].corr()

plt.figure(figsize=(12, 10))
sns.heatmap(correlation_matrix[['status']].sort_values('status', ascending=False), 
            annot=True, cmap='coolwarm', center=0)
plt.title('Feature Correlation with Target')
plt.tight_layout()
plt.show()

# Feature importance (using Random Forest)
X = df.drop('status', axis=1)
y = df['status'].map({'phishing': 1, 'legitimate': 0})

# Select only numeric columns for initial analysis
X_numeric = X.select_dtypes(include=[np.number])

# Train a quick model for feature importance
X_train, X_test, y_train, y_test = train_test_split(X_numeric, y, test_size=0.2, random_state=42)
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

# Get feature importance
feature_importance = pd.DataFrame({
    'feature': X_numeric.columns,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)

print("\nTop 10 Most Important Features:")
print(feature_importance.head(10))

# Plot feature importance
plt.figure(figsize=(12, 8))
sns.barplot(data=feature_importance.head(20), x='importance', y='feature')
plt.title('Top 20 Most Important Features')
plt.tight_layout()
plt.show()

# Model performance
y_pred = rf.predict(X_test)
print(f"\nAccuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"\nClassification Report:\n{classification_report(y_test, y_pred)}")

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()